# Llama 3.2-1B Activation-Based Estimation

Notebook này đã được viết lại theo repo sau refactor, dùng namespace mới `carve_lm`.

Nó chạy được trên Google Colab, Kaggle hoặc local nếu bạn mở notebook ngay trong repo.

Flow chính:
- bootstrap repo và dependencies
- load `meta-llama/Llama-3.2-1B`
- tạo calibration dataloader
- chạy activation-based estimation cho attention heads, attention groups, MLP neurons và embedding channels
- lưu score ra `artifacts/activation/llama32_1b_activation_scores.pt`


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/fannam/SoICT-LLM-Pruner.git"
REPO_NAME = "SoICT-LLM-Pruner"


def detect_runtime() -> str:
    try:
        import google.colab  # type: ignore
        return "colab"
    except ImportError:
        pass

    if Path("/kaggle/working").exists():
        return "kaggle"

    return "local"


def find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "carve_lm").exists():
            return candidate
    return None


runtime = detect_runtime()
repo_root = find_repo_root(Path.cwd())

if repo_root is None:
    if runtime == "colab":
        base_dir = Path("/content")
    elif runtime == "kaggle":
        base_dir = Path("/kaggle/working")
    else:
        base_dir = Path.cwd()

    repo_root = base_dir / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

os.chdir(repo_root)

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,notebooks]"],
    cwd=repo_root,
    check=True,
)

print({"runtime": runtime, "repo_root": str(repo_root)})


In [ ]:
hf_token = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
)

print("HF token found in environment:", bool(hf_token))
print("Nếu chưa có token hoặc model báo 401/403, chạy cell login bên dưới.")


In [ ]:
from huggingface_hub import notebook_login

if hf_token:
    print("HF token already found in environment. Skip interactive login.")
else:
    notebook_login()


In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding

from carve_lm.core import calculate_embedding_channels_global_score
from carve_lm.estimators import available_estimators, create_estimator
from carve_lm.pruners import available_element_pruning_strategies, create_pruner

os.environ["TOKENIZERS_PARALLELISM"] = "false"
auth_kwargs = {"token": hf_token} if hf_token else {}

MODEL_NAME = "meta-llama/Llama-3.2-1B"
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "validation"
MAX_SAMPLES = 64
MAX_LENGTH = 256
BATCH_SIZE = 4
AGGREGATION = "l2"
SEED = 13

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if device == "cuda" else torch.float32
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

print(
    {
        "device": device,
        "dtype": str(dtype),
        "model": MODEL_NAME,
        "max_samples": MAX_SAMPLES,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
    }
)

print("Available element estimators:", available_estimators("element."))
print("Available element pruning strategies:", available_element_pruning_strategies())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **auth_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    **auth_kwargs,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(model.__class__.__name__)
print(
    {
        "hidden_size": model.config.hidden_size,
        "num_layers": model.config.num_hidden_layers,
        "num_heads": model.config.num_attention_heads,
        "num_key_value_heads": getattr(model.config, "num_key_value_heads", model.config.num_attention_heads),
        "intermediate_size": model.config.intermediate_size,
    }
)


## Build calibration dataloader

Estimator activation của repo mới vẫn nhận `dataloader` ở từng hàm `estimate_*`. Ở đây dùng một calibration set nhỏ từ WikiText-2 để giảm thời gian chạy trên Colab/Kaggle.


In [ ]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=SPLIT)
text_column = "text"

raw_dataset = raw_dataset.filter(
    lambda example: example[text_column] is not None and example[text_column].strip() != ""
)

sample_count = min(MAX_SAMPLES, len(raw_dataset))
calibration_dataset = raw_dataset.shuffle(seed=SEED).select(range(sample_count))


def tokenize_batch(batch):
    return tokenizer(batch[text_column], truncation=True, max_length=MAX_LENGTH)


tokenized_dataset = calibration_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=calibration_dataset.column_names,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8 if device == "cuda" else None,
    return_tensors="pt",
)

calibration_loader = DataLoader(
    tokenized_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

first_batch = next(iter(calibration_loader))
print(f"Calibration samples: {sample_count}")
{key: tuple(value.shape) for key, value in first_batch.items()}


In [ ]:
def topk_summary(score_dict, topk=5):
    summary = {}
    for key, scores in score_dict.items():
        values, indices = torch.topk(scores, k=min(topk, scores.numel()))
        summary[key] = [
            {"index": int(index), "score": float(value)}
            for value, index in zip(values.tolist(), indices.tolist())
        ]
    return summary


def preview_mapping(mapping, limit=3):
    return dict(list(mapping.items())[:limit])


In [ ]:
estimator = create_estimator("element.activation", model, device=device)
pruner = create_pruner("element", model, device=device)

print(type(estimator).__name__)
print(type(pruner).__name__)


## Estimate attention head importance


In [ ]:
head_importance = estimator.estimate_attention_heads(calibration_loader, agg=AGGREGATION)
print(len(head_importance), head_importance[0].shape)


In [ ]:
head_topk = topk_summary(head_importance, topk=5)
preview_mapping(head_topk, limit=4)


## Estimate attention group importance

Repo mới có `estimate_attention_groups()` để score theo KV group trực tiếp. Với GQA/MQA, nên dùng score này nếu bạn prune theo group thay vì suy ra từ head score như flow cũ.


In [ ]:
group_importance = estimator.estimate_attention_groups(calibration_loader, agg=AGGREGATION)
print(len(group_importance), group_importance[0].shape)


In [ ]:
group_topk = topk_summary(group_importance, topk=5)
preview_mapping(group_topk, limit=4)


## Estimate MLP neuron importance


In [ ]:
neuron_importance = estimator.estimate_mlp_neurons(calibration_loader, agg=AGGREGATION)
print(len(neuron_importance), neuron_importance[0].shape)


In [ ]:
neuron_topk = topk_summary(neuron_importance, topk=5)
preview_mapping(neuron_topk, limit=4)


## Estimate embedding channel importance


In [ ]:
embedding_importance = estimator.estimate_embedding_channels(calibration_loader, agg=AGGREGATION)
print(len(embedding_importance), next(iter(embedding_importance.values())).shape)


In [ ]:
embedding_global_score = calculate_embedding_channels_global_score(embedding_importance)
embedding_values, embedding_indices = torch.topk(
    embedding_global_score,
    k=min(20, embedding_global_score.numel()),
)
embedding_topk = [
    {"channel": int(index), "score": float(value)}
    for value, index in zip(embedding_values.tolist(), embedding_indices.tolist())
]

print("Per-hook tensors:", list(embedding_importance.keys())[:5], "...")
embedding_topk[:10]


In [ ]:
output_path = repo_root / "artifacts" / "activation" / "llama32_1b_activation_scores.pt"
output_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_name": MODEL_NAME,
        "aggregation": AGGREGATION,
        "dataset_name": DATASET_NAME,
        "dataset_config": DATASET_CONFIG,
        "split": SPLIT,
        "max_samples": MAX_SAMPLES,
        "max_length": MAX_LENGTH,
        "head_importance": head_importance,
        "group_importance": group_importance,
        "neuron_importance": neuron_importance,
        "embedding_importance": embedding_importance,
        "embedding_global_score": embedding_global_score,
    },
    output_path,
)

output_path


## Optional: prune attention groups with the new API

Cell này minh họa flow prune theo repo mới. `prune_attention_group()` bây giờ nhận `group_importance` trực tiếp, đây là cách nên dùng cho Llama 3.2-1B.


In [ ]:
original_groups = getattr(model.config, "num_key_value_heads", model.config.num_attention_heads)
target_group = max(1, original_groups // 2)

if target_group < original_groups:
    pruned_model = pruner.prune_attention_group(
        group_importance=group_importance,
        target_group=target_group,
    )
    print(
        {
            "original_num_attention_heads": model.config.num_attention_heads,
            "original_num_key_value_heads": original_groups,
            "pruned_num_attention_heads": pruned_model.config.num_attention_heads,
            "pruned_num_key_value_heads": pruned_model.config.num_key_value_heads,
        }
    )
else:
    print("This model has no removable attention group under the chosen rule.")


Nếu bị out-of-memory trên Colab/Kaggle, giảm `MAX_SAMPLES`, `MAX_LENGTH` hoặc `BATCH_SIZE` rồi chạy lại từ cell load dataset trở xuống.
